To test the ApplianceRecommender, we should create a structured notebook that validates:

Filtering: Are recommendations restricted to the correct category and capacity?
Similarity: Do the results look technically similar to the input?
Value Logic: Are we seeing "Good Deals" in the recommendations?

In [1]:
import sys
from pathlib import Path

# Adjust path to import from src
root = Path.cwd()
while root != root.parent:
    if (root / "src").exists(): break
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
import numpy as np
from src.recommender import ApplianceRecommender
import warnings
warnings.filterwarnings("ignore")

# Initialize Recommender
# This might take a few seconds as it pre-calculates Fair Prices for the pool
recommender = ApplianceRecommender()

### **Test Case 1: Standard AC Recommendation**
We'll test with a 1.5 Ton 5-Star LG AC.

In [9]:
complete_test_features_dict = {'rating': 4.7,
 'category': 'Refrigerator',
 'n_features': 10,
 'brand_name': 'lg',
 'star_rating': 2.0,
 'has_star_rating': 1,
 'has_inverter': 1,
 'has_wifi': 0,
 'has_voice_control': 0,
 'has_app_control': 0,
 'smart_connectivity_score': 0,
 'ac_split': 0,
 'ac_window': 0,
 'ac_pm25_filter': 0,
 'ac_hepa_filter': 0,
 'ac_auto_clean': 0,
 'ac_hot_and_cold': 0,
 'ac_copper_condenser': 0,
 'ac_Dehumidification': 0,
 'ac_Turbo Mode': 0,
 'ac_Self Diagnosis': 0,
 'ref_single_door': 0,
 'ref_multi_door': 1,
 'ref_chest_freezer': 0,
 'ref_side_door': 0,
 'ref_french_door': 0,
 'ref_double_door': 1,
 'ref_triple_door': 0,
 'ref_frost_free': 1,
 'ref_convertible': 1,
 'ref_door_alarm': 0,
 'ref_door_lock': 0,
 'ref_dispenser': 0,
 'ref_door_display': 0,
 'ref_mini': 0,
 'wm_fully_automatic': 0,
 'wm_semi_automatic': 0,
 'wm_with_dryer': 0,
 'wm_washer_only': 0,
 'wm_dryer_only': 0,
 'wm_front_load': 0,
 'wm_top_load': 0,
 'wm_inbuilt_heater': 0,
 'wm_quick_wash': 0,
 'wm_ss_tub': 0,
 'wm_child_lock': 0,
 'wm_shock_proof': 0,
 'wm_display': 0,
 'capacity_sq': 117649.0,
 'n_features_sq': 100,
 'rating_sq': 22.090000000000003,
 'capacity_n_features': 3430.0,
 'capacity_rating': 1612.1000000000001,
 'rating_n_features': 47.0,
 'smart_intensity': 0,
 'ac_premium_features': 0,
 'ref_door_complexity': 0,
 'wm_tech_level': 0,
 'features_above_avg': 1.7261904761904763,
 'rating_above_avg': 0.33908730158730194,
 'capacity_above_avg': 56.54960317460319,
 'brand_frequency': 261,
 'capacity_ac_tons': 0.0,
 'capacity_value': 343,
 'capacity_wm_kg': 0.0}

In [13]:
# Load actual training data
x_df = pd.read_parquet(
    r"D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/X_train.parquet"
)

y_df = pd.read_parquet(
    r"D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/y_train.parquet"
)
sample_row_idx = 1111 # chossing a different index for variety
# complete_test_features_dict = x_df.loc[sample_row_idx].to_dict()

observed_price_for_complete_test = float(np.expm1(y_df.loc[sample_row_idx, 'log_price']))

recs_ac = recommender.get_recommendations(complete_test_features_dict, n=30)
recs_df = pd.DataFrame(recs_ac)

# Display relevant columns to verify
print(f'Original Price:{observed_price_for_complete_test}')
cols_to_show = ['brand_name', 'star_rating', 'capacity_ac_tons', 'actual_price', 'fair_price', 'value_score']
print("Recommendations for 1.5 Ton LG AC:")
recs_df[cols_to_show]

Original Price:48589.999999999985
Recommendations for 1.5 Ton LG AC:


,brand_name,star_rating,capacity_ac_tons,actual_price,fair_price,value_score
0,lg,3.0,0.0,38990.0,44792.523438,1.148821
1,lg,2.0,0.0,44990.0,47458.750000,1.054873
2,lg,2.0,0.0,48590.0,48552.820312,0.999235
3,lg,2.0,0.0,51490.0,47628.554688,0.925006


In [14]:
test_input_ac = {
    'category': 'Refrigerator',
    'brand_name': 'voltas',
    'capacity_value': 300,
    'star_rating': 4,
    'has_inverter': 0,
    'ac_split': 0
}

recs_ac = recommender.get_recommendations(test_input_ac, n=30)
recs_df = pd.DataFrame(recs_ac)

# Display relevant columns to verify
cols_to_show = ['brand_name', 'star_rating', 'capacity_ac_tons', 'actual_price', 'fair_price', 'value_score']
print("Recommendations for 1.5 Ton LG AC:")
recs_df[cols_to_show]

Recommendations for 1.5 Ton LG AC:


,brand_name,star_rating,capacity_ac_tons,actual_price,fair_price,value_score
0,godrej,5.0,0.0,23765.0,27533.921875,1.158591
1,blue star,3.0,0.0,26800.0,30025.179688,1.120343
2,blue star,3.0,0.0,28000.0,30282.562500,1.081520
3,godrej,3.0,0.0,24490.0,26123.816406,1.066714
4,godrej,4.0,0.0,24500.0,25597.000000,1.044776
5,voltas,3.0,0.0,35008.0,32830.328125,0.937795
6,whirlpool,5.0,0.0,33940.0,31885.193359,0.939458


### **Test Case 2: "Value Hunter" Check**
Let's see if the recommender finds products where value_score > 1.1 (meaning the Fair Price is 10%+ higher than the Market Price).

In [15]:
# Check the top recommendation's value status
top_rec = recs_ac[0]
print(f"Top Recommendation Value Score: {top_rec['value_score']:.2f}")
if top_rec['value_score'] > 1.0:
    print("✅ Success: Found a product priced below its fair value.")
else:
    print("ℹ️ Info: Found a close spec match, but not necessarily a discount.")

Top Recommendation Value Score: 1.56
✅ Success: Found a product priced below its fair value.


### **Test Case 3: Cross-Category (Refrigerator)**

In [15]:
test_input_ref = {
    'category': 'WashingMachine',
    'brand_name': 'godrej',
    'capacity_value':15,
    'capacity_ref_litres': 1,
    'ref_double_door': 1,
    'ref_frost_free': 1
}

recs_ref = pd.DataFrame(recommender.get_recommendations(test_input_ref, n=3))
print("\nRecommendations for 250L Samsung Refrigerator:")
recs_ref[['brand_name', 'capacity_ref_liters', 'actual_price', 'fair_price', 'value_score']]


Recommendations for 250L Samsung Refrigerator:


,brand_name,capacity_ref_liters,actual_price,fair_price,value_score
0,lg,0.0,59994.0,64731.960938,1.078974


In [16]:
recs_ref

,rating,category,n_features,brand_name,star_rating,has_star_rating,has_inverter,has_wifi,has_voice_control,has_app_control,...,value_score,similarity_score,recommendation_score,_comp_capacity,_comp_brand,_comp_star,_comp_inverter,_comp_feature,capacity_mode,explanation
0,4.7,WashingMachine,11,lg,0.0,0,1,1,0,1,...,1.078974,0.673222,0.612594,1.0,0.0,0.5,1.0,0.982217,exact,"{'match_reasons': 'Smart: WiFi, App control', ..."
